# 第七讲 第一个可重现实验

固定公开合成数据，训练36、验证12、测试12。先阅读实验指南，再从上到下运行。最终重启内核并运行全部；本实验不提供真实盲测或市场预测证据。

## 1 读取固定材料
默认路径来自模块文件，不输出使用者的完整个人路径。

In [1]:
from pathlib import Path
from tempfile import TemporaryDirectory
from fractions import Fraction
import json
import numpy as np
import experiment as exp

groups, config = exp.load_materials(exp.BASE / "data", exp.BASE / "config.json")
print({role: len(rows) for role, rows in groups.items()})
print("候选斜率：", config["candidate_weights"])
print("候选截距：", config["candidate_biases"])

{'train': 36, 'validation': 12, 'test': 12}
候选斜率： [1.5, 2.0, 2.5]
候选截距： [0.0, 10.0, 20.0]


## 2 核对编号身份
三个角色不能相交，也不能遗漏六十条中的任何一条。

In [2]:
role_ids = {role: {r["house_id"] for r in rows} for role, rows in groups.items()}
print({role: sorted(ids) for role, ids in role_ids.items()})
assert len(set.union(*role_ids.values())) == 60
assert not (role_ids["train"] & role_ids["validation"])
assert not (role_ids["train"] & role_ids["test"])
assert not (role_ids["validation"] & role_ids["test"])

{'train': ['H001', 'H002', 'H004', 'H005', 'H007', 'H010', 'H011', 'H013', 'H014', 'H015', 'H017', 'H020', 'H021', 'H023', 'H025', 'H027', 'H028', 'H029', 'H033', 'H036', 'H037', 'H038', 'H040', 'H041', 'H043', 'H045', 'H047', 'H048', 'H050', 'H051', 'H052', 'H054', 'H055', 'H058', 'H059', 'H060'], 'validation': ['H008', 'H009', 'H016', 'H018', 'H019', 'H030', 'H031', 'H034', 'H046', 'H053', 'H056', 'H057'], 'test': ['H003', 'H006', 'H012', 'H022', 'H024', 'H026', 'H032', 'H035', 'H039', 'H042', 'H044', 'H049']}


## 3 用训练数据拟合两个预定方案
函数只接收训练记录，候选网格与均值基线事先固定。

In [3]:
models, training_record = exp.fit_models(groups["train"], config)
print(models)
for row in training_record["grid_scores"]:
    print(row)

[{'name': 'constant', 'value_wan': 152.19444444444446}, {'name': 'linear_grid', 'weight': 2.0, 'bias': 10.0}]
{'weight': 1.5, 'bias': 0.0, 'training_mae_wan': 46.52777777777778}
{'weight': 1.5, 'bias': 10.0, 'training_mae_wan': 36.52777777777778}
{'weight': 1.5, 'bias': 20.0, 'training_mae_wan': 26.52777777777778}
{'weight': 2.0, 'bias': 0.0, 'training_mae_wan': 11.305555555555555}
{'weight': 2.0, 'bias': 10.0, 'training_mae_wan': 4.638888888888889}
{'weight': 2.0, 'bias': 20.0, 'training_mae_wan': 8.694444444444445}
{'weight': 2.5, 'bias': 0.0, 'training_mae_wan': 23.916666666666668}
{'weight': 2.5, 'bias': 10.0, 'training_mae_wan': 33.916666666666664}
{'weight': 2.5, 'bias': 20.0, 'training_mae_wan': 43.916666666666664}


## 4 验证集只比较已拟合方案
这一步参与开发，不能再把验证得分称为完全未见证据。

In [4]:
selected, validation_scores = exp.select_model(models, groups["validation"], config["tolerance"])
print(validation_scores)
print("冻结选择：", selected)
assert selected == {"name": "linear_grid", "weight": 2.0, "bias": 10.0}

[{'name': 'constant', 'validation_mae_wan': 41.319444444444436}, {'name': 'linear_grid', 'validation_mae_wan': 4.75}]
冻结选择： {'name': 'linear_grid', 'weight': 2.0, 'bias': 10.0}


## 5 冻结之后评价教学测试
两个方法在相同十二条记录上比较。

In [5]:
test_rows, test_mae = exp.score_records(selected, groups["test"])
baseline_rows, baseline_mae = exp.score_records(models[0], groups["test"])
print("测试 MAE：", test_mae)
print("固定基线测试 MAE：", baseline_mae)
for row in test_rows:
    print(row)

测试 MAE： 4.666666666666667
固定基线测试 MAE： 32.333333333333336
{'house_id': 'H003', 'prediction_wan': 142.0, 'label_wan': 145.0, 'absolute_error_wan': 3.0}
{'house_id': 'H006', 'prediction_wan': 166.0, 'label_wan': 161.0, 'absolute_error_wan': 5.0}
{'house_id': 'H012', 'prediction_wan': 204.0, 'label_wan': 197.0, 'absolute_error_wan': 7.0}
{'house_id': 'H022', 'prediction_wan': 90.0, 'label_wan': 85.0, 'absolute_error_wan': 5.0}
{'house_id': 'H024', 'prediction_wan': 100.0, 'label_wan': 92.0, 'absolute_error_wan': 8.0}
{'house_id': 'H026', 'prediction_wan': 144.0, 'label_wan': 147.0, 'absolute_error_wan': 3.0}
{'house_id': 'H032', 'prediction_wan': 192.0, 'label_wan': 191.0, 'absolute_error_wan': 1.0}
{'house_id': 'H035', 'prediction_wan': 120.0, 'label_wan': 124.0, 'absolute_error_wan': 4.0}
{'house_id': 'H039', 'prediction_wan': 100.0, 'label_wan': 100.0, 'absolute_error_wan': 0.0}
{'house_id': 'H042', 'prediction_wan': 178.0, 'label_wan': 185.0, 'absolute_error_wan': 7.0}
{'house_id': 'H0

## 6 用独立有理数算术核对
从原始整数记录直接求误差和，不调用核心 MAE 函数。Fraction 精确保留本题分数，避免把程序输出本身当唯一参考。

In [6]:
training_mean = Fraction(sum(int(r["price_wan"]) for r in groups["train"]), 36)
integer_error_sum = sum(abs(2*int(r["area_m2"])+10-int(r["price_wan"])) for r in groups["test"])
constant_error_sum = sum(abs(training_mean-int(r["price_wan"])) for r in groups["test"])
print("训练均值：", training_mean)
print("两种误差和：", integer_error_sum, constant_error_sum)
assert training_mean == Fraction(5479,36)
assert integer_error_sum == 56
assert constant_error_sum == 388
assert abs(test_mae-float(Fraction(14,3))) < 1e-12
assert abs(baseline_mae-float(Fraction(97,3))) < 1e-12

训练均值： 5479/36
两种误差和： 56 388


## 7 完整记录与版本摘要
摘要帮助核对字节版本，不证明数据真实性。

In [7]:
record = exp.run()
print(record["inputs_sha256"])
print(record["environment"])
print("固定参考核验：", exp.check_reference(record))

{'houses.csv': '592cb539089de8661e39c57c1f148fc2a5f73aaa1f93ed92ad25554568968e85', 'split.csv': '9e7f29d99da6b1ab7077f16d0083fd02b5207e0fe34f578f1a0cb8032f6c75fc', 'config.json': '5bc22bf6eb1428d214176555ca2d714979fc8c9ab1bb0a3ad9d60e4887f89635'}
{'python': '3.12.14', 'numpy': '2.3.5', 'device': 'CPU'}
固定参考核验： True


## 8 两次完整调用的记录一致性
这里是同一内核中的调用对照；命令行新进程复跑另在实验指南完成，不能把两者混称。

In [8]:
with TemporaryDirectory() as temporary:
    first = exp.run(output=Path(temporary)/"first")
    second = exp.run(output=Path(temporary)/"second")
    first_bytes = (Path(temporary)/"first"/"result.json").read_bytes()
    second_bytes = (Path(temporary)/"second"/"result.json").read_bytes()
    print("记录相同：", first_bytes == second_bytes)
    assert first == second
    assert first_bytes == second_bytes

记录相同： True


## 9 边界与信息流检查
包含改变测试标签仍不改变训练与验证选择，遗漏编号拒绝，形状与有限性检查。

In [9]:
exp.self_test()

PASS: reference, split, translation, test-isolation, config-contract, missing-ID and shape/finite checks


## 10 写下结论与边界
请用两句话说明这次能够复现什么，以及尚不能证明什么。不要把同一数据重跑两次当成两份独立统计证据。